# Introduction

This tutorial shows how to run a very basic ASCOT5 simulation.
While everything 
In this tutorial we will run a short simulation that traces 100 markers collisionlessly in a tokamak magnetic field.
Even though everything is simplified here, 


<div class="alert alert-block alert-warning">
<b>Example:</b> Use yellow boxes for examples that are not 
inside code cells, or use for mathematical formulas if needed.
</div>

<div class="alert alert-block alert-info">
<b>Tip:</b> Use blue boxes (alert-info) for tips and notes. 
If it’s a note, you don’t have to include the word “Note”.
</div>

<a id='stepbystep'></a>

## Getting started

Initialize ``Ascot`` object which will store the simulation data and is used to execute the simulations.

In [ ]:
from a5py import Ascot

a5 = Ascot()

``Ascot`` has an attribute ``data`` that is used to access the simulation data.
Currently it is empty.

In [ ]:
a5.data.show_contents()


Initializing ``Ascot`` without providing filename means all data is kept in memory, and so is lost when the Python session expires.
ASCOT5 usually deals with data that is of several gigabytes, and the results may require significant computations, so storing the data to file is often desired:

In [ ]:
a5_stored_in_disk = Ascot("ascot.h5", create=True)
!ls | grep "ascot.h5"

``create=True`` is required to create a new file to prevent accidentally overwriting existing data.
Remove it when working with a file that already exists.

## Generating inputs

The inputs that are required for a simulation depends on what physics are included.
Here it's necessary to only provide markers that are traced and the magnetic field.

**There are two ways to create inputs.**
The ``data`` attribute is an instance of class ``AscotData`` that also manages the input creation:

In [ ]:
print(a5.data.__class__.__doc__)

**The first way is to directly use the methods of this class.**
For example, the (guiding center) markers can be created in a following way:

In [ ]:
import unyt
import numpy as np

# 100 alpha particle guiding centers distributed evenly on the outer-mid plane
# with uniform pitch distribution
nmrk = 100
a5.data.create_guidingcentermarker(
    species="alpha",
    r=(6.2 + (8. - 6.2) * np.random.rand(nmrk))*unyt.m,
    z=0.*unyt.m,
    ekin= 3.5e6*unyt.eV,
    pitch=1. - 2 * np.random.rand(nmrk),
)

The method contains the documentation on what data is needed to create the corresponding inputs.

In [ ]:
help(a5.data.create_guidingcentermarker)

However, this is the low-level interface.
ASCOT5 has been interfaced to many other codes and data formats, and these high-level interfaces are referred to as input ``templates``, and using these is **the second way of creating inputs**.
The templates also contain tools to create inputs e.g. from analytical forms.

Here we use the template for analytical tokamak field to create ITER-like magnetic field input.

In [ ]:
from a5py.templates import PremadeMagneticField
template = PremadeMagneticField(a5, field="iter-baseline")
template.create_input()

Detailed information on how to use a specific template can be found in its documentation:

In [ ]:
print(PremadeMagneticField.__doc__)

Now all data necessary for running the simulation exists.

In [ ]:
a5.data.show_contents()

## Executing simulations

First define the parameters that specify the simulation.

In [ ]:
from a5py import SimulationOptions

opt = SimulationOptions.from_dict(
    simulation={"mode": "guiding-center", "timestep": 1e-8},
    physics={"enable_orbit_following": True},
    endconditions={
        "activate_simulation_time_limits": True,
        "activate_real_time_limit": True,
        "max_mileage": 1e-6,
        "max_real_time": 10,
        },
)

The parameters are divided to different groups:

- ``simulation`` specifies the simulation mode, time-step, and other general settings.
- ``physics`` controls what physics are included in the simulation.
- ``endconditions`` specify the conditions when the simulation of a marker is finished.

There are two additional groups which set the simulation output.
In every simulation, the initial and final *state* of the markers is stored but in addition it is possible to record marker *orbit* in between.

Collecting exact phase-space position of large number of markers for the duration of the whole simulation is most of the time unfeasible, which is why instead particle *histograms* can be collected as well.
These histograms represent the particle distribution function and can be used to compute various moments in post-processing.

``SimulationOptions`` can be modified after creation.

In [ ]:
opt.orbit.collect = "interval"
opt.orbit.buffer_size = 1000
opt.orbit.interval = 1e-7

opt.histograms = [
    {
        "dimensions": [
            ("r", 4.*unyt.m, 6.*unyt.m, 10),
            ("z", -4.*unyt.m, 4.*unyt.m, 10),
            ],
        "charge_interval": (2,2),
    }
]


Options are stored in the simulation output, but they can also be written to a text file and read from there.

In [ ]:
opt.write_toml("options.toml")
opt = SimulationOptions.from_toml("options.toml")

Now that ``orbit`` and ``histograms`` are set, we are ready to launch the simulation.
It's worth mentioning that the default values in options are set so that effectively everything is disabled (except options that would make the physics less realistic and so are rarely touched).



In [ ]:
run = a5.simulate(params=opt)

In [ ]:
run.getstate("r", "z", state="end")

In [ ]:
import unyt
import numpy as np
from a5py import Ascot, SimulationOptions
from a5py.templates import PremadeMagneticField

a5 = Ascot()

opt = SimulationOptions.from_dict(
    simulation={"mode": "gyro-orbit", "timestep": 1e-8, "enable_adaptive": True},
    physics={"enable_orbit_following": True},
    endconditions={
        "activate_simulation_time_limits": True,
        "activate_real_time_limit": True,
        "max_mileage": 1e-4,
        "max_real_time": 1.e1,
        },
)
opt.orbit.collect = "interval"
opt.orbit.buffer_size = 1000
opt.orbit.interval = 1e-7

opt.histograms = [
    {
        "dimensions": [
            ("r", 4.*unyt.m, 8.*unyt.m, 10),
            #("z", -4.*unyt.m, 4.*unyt.m, 10),
            ],
        "charge_interval": (2,2),
    }
]

template = PremadeMagneticField(a5, field="iter-baseline")
template.create_input()

nmrk = 1
a5.data.create_guidingcentermarker(
    species="alpha",
    r=(6.2 + (8. - 6.2) * np.random.rand(nmrk))*unyt.m,
    z=0.*unyt.m,
    ekin= 3.5e6*unyt.eV,
    pitch=1. - 2 * np.random.rand(nmrk),
)

run = a5.simulate(params=opt)

r, z = run.getorbit("r", "z", filter=[1])
dist = run.getdist(0, "r")

import matplotlib.pyplot as plt
plt.plot(dist.dimensions["r"], dist.distribution)

## Post-processing

## Need help?

1. Ask in our ASCOT5 Slack channel.

2. If you have an issue to report, use the GitHub issue tracker.
   For bugs, state which version/branch you are using and try to provide the HDF5 file.

3. Join one of our "weekly" meetings to present your research and discuss any issues.